# Plugging in a ZMART-driver

A driver is what lets the controller talk to one particular microscope. Writing one means filling in a class, `ZmartDriver`, with one method per command. This notebook writes a driver for a pretend microscope that lives in memory, checks that it fits, installs it, and drives it like any other. Run the cells from top to bottom. The [README](README.md) says exactly what every method must hand back.

## The two files

A driver is a folder with two files. `zmart_driver.json` holds the driver's name and how to reach the microscope; it is the file you point at to install the driver. The pretend microscope needs nothing to be reached, so its connection only says which instrument it is.

In [ ]:
from pathlib import Path

Path("pretend_scope").mkdir(exist_ok=True)

In [ ]:
%%writefile pretend_scope/zmart_driver.json
{
  "name": "pretend-scope",
  "driver": "zmart_driver.py",
  "connection": {
    "microscope": "pretend-01"
  }
}

`zmart_driver.py` holds the class. Making it opens the connection to the vendor software; here it just sets up a stage and an exposure in memory. Each method does one command and hands back `True` and the values, or `False` and a message that says why not. The controller turns that into the `{"success": ..., "content": ...}` answer a workflow sees. Notice that `set_xyz` checks the travel before moving, and that `acquire` says where it saved the picture and where on the sample it was taken: that is how a workflow finds its pictures on any microscope.

In [ ]:
%%writefile pretend_scope/zmart_driver.py
import json
from pathlib import Path

TRAVEL = {"x": [-1000.0, 1000.0], "y": [-1000.0, 1000.0], "z": [-100.0, 100.0]}   # micrometres from the origin
HALF_FIELD = 50.0                                                                   # half a field of view


class ZmartDriver:
    def __init__(self, connection):
        self.position = {"x": 0.0, "y": 0.0, "z": 0.0}
        self.settings = {"exposure_ms": 10.0}
        self.output_root = Path("pretend_scope_images")
        self.output_root.mkdir(exist_ok=True)

    def disconnect(self):
        pass

    def get_info(self):
        return True, "A pretend microscope in memory. exposure_ms is the exposure time in milliseconds. +z points up."

    def get_actuators(self):
        return True, (["motor"], ["motor"], ["motor"])

    def get_xyz(self, with_actuators):
        p = self.position
        canvas = (TRAVEL["x"][0] - HALF_FIELD, TRAVEL["x"][1] + HALF_FIELD,
                  TRAVEL["y"][0] - HALF_FIELD, TRAVEL["y"][1] + HALF_FIELD,
                  TRAVEL["z"][0], TRAVEL["z"][1])
        return True, (p["x"], p["y"], p["z"], "motor", "motor", "motor", canvas)

    def set_xyz(self, x, y, z, with_actuators):
        for axis, value in {"x": x, "y": y, "z": z}.items():
            low, high = TRAVEL[axis]
            if not low <= value <= high:
                return False, f"{axis} = {value} is outside the travel, {low} to {high}"
        self.position = {"x": x, "y": y, "z": z}
        return True, ("motor", "motor", "motor")

    def get_state(self):
        return True, (dict(self.settings), {"objective": "10x"})

    def set_state(self, changeable):
        for name in changeable:
            if name not in self.settings:
                return False, f"unknown setting {name}"
        self.settings.update(changeable)
        return True, dict(changeable)

    def get_acquisition_settings(self):
        return True, {"format": {"options": ["json"], "active": "json"}}

    def acquire(self, position_label, acquisition_settings):
        path = self.output_root / f"{position_label}.json"
        path.write_text(json.dumps({"position": self.position, "settings": self.settings}))
        p = self.position
        plane = {"path": str(path), "c": 0, "z": 0, "t": 0, "x_um": p["x"], "y_um": p["y"], "z_um": p["z"]}
        return True, ([str(path)], [plane])

    def get_procedures(self):
        return True, {}

    def run_procedure(self, procedure):
        return False, f"unknown procedure {procedure.get('name')}"

## Check that it fits

`validate_driver` connects, calls every `get_*` method, and compares each answer with what the controller expects. It returns the problems it found, one plain sentence each. An empty list means the driver fits. This is the loop to work in while you write a driver: write a method, validate, read the problems, repeat.

In [ ]:
from zmart_controller import mic

mic.validate_driver("pretend_scope/zmart_driver.json")

## Install it

Installing means pointing the controller at the driver's `zmart_driver.json`, once. The controller writes it into its registry on this computer, and from then on `get_instruments` knows it and any Python session can connect to it by name.

In [ ]:
mic.register_driver("pretend_scope/zmart_driver.json")
mic.get_instruments()

## Drive it

Now the pretend microscope answers every command exactly like the mock did in Part 1. The "picture" is a small text file that records where and how it was taken. The move outside the travel comes back as `success: False`, with the message your own `set_xyz` wrote.

In [ ]:
mic.connect("pretend-scope")
mic.set_xyz(100, 50, 0)
mic.acquire(position_label="A1")["content"]["files"]

In [ ]:
mic.set_xyz(5000, 0, 0)

In [ ]:
mic.disconnect()

## Clean up

This driver does not need to stay installed. For a real microscope, each method does the same job but by talking to the vendor software, and the driver stays installed on the microscope computer.

In [ ]:
mic.remove_driver("pretend-scope")
mic.get_instruments()

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.